# Outfit Generator — Co-occurrence Recommendation System

**Authors:** Keyan Saguibo, Hana Tanisaka, Aarnav Dharia, Itzel Valdez (NSDC S26)

This notebook builds an outfit recommendation system from the [Marqo/Polyvore](https://huggingface.co/datasets/Marqo/polyvore) dataset (~94k clothing items grouped into outfits). Given one clothing item, it recommends other items that complete a cohesive outfit.

**Approach.** We use a *co-occurrence* recommender — a count-based collaborative-filtering baseline. We learn which items tend to appear together in real outfits, then recommend the items most frequently paired with the user's input. We build two versions:

- **Model 1 (category-level):** learns pairings between broad categories (`Jeans` ↔ `Shoes`). Simple, interpretable baseline.
- **Model 2 (sub-genre-level):** learns pairings between finer style labels (`Skinny Jeans` ↔ `Platform Shoes`) and blocks same-category recommendations so it never suggests pants when you already have a dress.

**Accessibility motivation.** The system recommends by item *identity*, not color, so it works for colorblind users who can't rely on color matching.

**Reproducibility.** This notebook runs top-to-bottom from the public HuggingFace dataset — no manual downloads or Google Drive needed.

---


## 0. Setup

In [ ]:
# Install dependencies (Colab already has pandas/sklearn/matplotlib)
!pip install -q datasets

In [ ]:
from datasets import load_dataset
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import random
import io
import ast

RANDOM_STATE = 42  # one seed used everywhere so results are reproducible
random.seed(RANDOM_STATE)

## 1. Load the data

We load directly from HuggingFace. Each row is one clothing item with an image, a category, a free-text description, and an `item_ID`. The part of `item_ID` before the underscore is the **outfit ID** — items sharing it belong to the same outfit.

In [ ]:
dataset = load_dataset("Marqo/polyvore")
df = dataset["data"].to_pandas()
print("Raw shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# How many distinct categories does the raw data contain?
print("Unique raw categories:", df["category"].nunique())

## 2. Clean the data

The raw dataset mixes fashion with furniture, makeup, home decor, etc. We do three things:

1. **Remove** non-clothing and out-of-scope categories (~260 of them).
2. **Consolidate** fine-grained labels into broad categories (`Boyfriend Jeans` → `Jeans`, `Sandals` → `Shoes`).
3. **Derive `outfit_id`** from `item_ID`, then drop outfits with only one item (you can't learn a pairing from a single item).

In [ ]:
# --- 2a. Categories to remove entirely (non-clothing / out of scope) ---
remove_categories = {
    'Sunglasses', 'Font', 'Watches', 'Makeup', 'Floral Decor', 'Accessories',
    'Dining Chairs', 'Accent Tables', 'Toys', 'Food & Drink', 'Bags',
    'Bracelets & Bangles', 'Fragrance', 'Clutches', 'Home Decor', 'Eyeliner',
    'Electronics', 'Nail Polish', 'Beauty Products', 'Lipstick', 'Rings',
    'Intimates', 'Drinkware', 'Tote Bags', 'Face Powder', 'Accent Chairs',
    'Sofas', 'Mascara', 'Eyeshadow', 'Blush', 'Bar Cabinets', 'Floor Lamps',
    'Kitchen & Dining', 'Mirrors', 'Belts', 'Hats', 'Eye Makeup', 'Home Decor',
    'Haircare', 'Dressers', 'Table Lamps', 'Throw Pillows', 'Wallpaper',
    'Clothing', "Men's Clothing", 'Ceiling Lights', 'Rugs', 'Vases', 'Clocks',
    'Stationery', 'Kids', 'Sideboards', 'Lighting', 'Tights', 'Makeup Brushes',
    'Wall Lights', 'Sports Accessories', "Men's Umbrellas", 'Gift Cards',
    "Men's Hair Care", 'Curtain Rods', 'Sleepwear', 'Lip Gloss', 'Wallets',
    'Outerwear', 'Small Storage', 'Shower Curtain', 'Blankets', 'Frames',
    'Office Accessories', 'Candles & Candleholders', 'Messenger Bags',
    'Foundation', 'Leggings', 'Pets', 'Hosiery', 'Jewelry', 'Shapewear',
    'Gloves', 'Body Art', 'Charms and Pendants', 'Nightstands', 'Serveware',
    'Benches', 'Media', 'Bath Accessories', 'Standing Fans', 'Eyeglasses',
    'Ottomans', 'Hair Styling Tools', 'Bras', 'Brooches', 'Bags & Cases',
    'Umbrellas', "Children's Bedding", 'False Eyelashes', 'Nail Treatment',
    'Costumes', 'Concealer', 'Outdoor Lighting', 'Outdoor Benches', 'Baby',
    'Lip Makeup', "Men's Sunglasses", 'Hair Conditioner', 'Books',
    'Body Moisturizers', 'Body Cleansers', 'Sports & Outdoors', 'Eyewear',
    'Furniture', 'Beds', 'Chemises', 'Juniors', 'Eyelash Curlers',
    'Gift Sets & Kits', 'Bikinis', 'Luggage', 'Panties & Thongs', 'Curtains',
    'Sun Care', 'Bikini Tops', 'Table Linens', 'Holiday Decorations',
    'One Piece Swimsuits', 'Cheek Bronzer', 'Bikini Bottoms', 'Makeup Primer',
    'Beach Towels', 'Nail Care', 'Bath & Body', 'Hair Shampoo', 'Dining Tables',
    'Boys', 'Cabinets', 'Sheets & Pillowcases', 'Sports Bras', 'Outdoor Decor',
    'Hair Color', 'Styling Products', 'Napkin Rings', "Men's Fashion",
    'Bar Tools', 'Beauty Accessories', "Men's Fragrance", 'Small Appliances',
    'Hammock & Swings', "Men's Tech Accessories", 'Pajamas', 'Outdoors',
    'Outdoor Stools', 'Bath Rugs & Mats', 'Armoires', "Men's Backpacks",
    'Briefcases', 'Face Cleansers', 'Panel Screens', 'Home Improvement',
    'Aprons', 'Dinnerware', 'Tinted Moisturizer', 'Makeup Remover',
    "Men's Necklace", 'Ties', 'Cleaning', 'Stools', 'Entertainment Units',
    'Fans', 'Bath Towels', 'Patio Furniture', 'Outdoor Chairs',
    'Kitchen Gadgets & Tools', 'Face Care', "Children's Decor",
    'Storage & Organization', "Men's Eyeglasses", "Men's Watches", 'Robes',
    'Blow Dryers & Irons', 'Brushes & Combs', 'Kitchen Linens', 'Duvet Covers',
    'Cover-ups', 'Bath', 'Lip Pencils', 'Handkerchiefs', "Men's Socks",
    "Men's Belts", 'Bed Accessories', 'Window Treatments', 'Bow Ties',
    'Patio Umbrellas', 'Home Fragrance', 'Outdoor Tables',
    'Outdoor Loungers & Day Beds', "Men's Underwear", 'Eye Care', 'Desks',
    'File Cabinets', 'Office Chairs', 'Desk Lamp', 'Barstools',
    'Outdoor Patio Sets', 'Cookbooks', 'Fireplace Accessories', 'Comforters',
    'Face Moisturizer', 'Bed Pillows', 'Party Supplies', "Men's Deodorant",
    'Cheek Makeup', 'Swimwear', 'Suspenders', "Children's Furniture",
    'Manicure Tools', 'Flooring', 'Sharpeners', "Men's Skincare",
    "Men's Shaving", "Men's Wallets", 'Window Blinds', 'Cutlery', 'Face Toners',
    'Teapots', 'Bedding', 'Decorative Hardware', 'Tweezers & Brow Tools',
    'Nursery Furniture', "Children's Room", 'Baby Bedding', 'Paint',
    "Men's Rings", "Men's Bracelets", 'Oral Care', 'Face Masks', 'Lip Stain',
    'Bakeware', "Men's Briefcases", 'Skincare', "Men's Accessories",
    "Men's Messenger Bags", 'Cookware', 'Quilts & Coverlets', 'Health',
    'Maternity', "Men's Scarves", "Men's Bags & Wallets", "Men's Gloves",
    "Men's Grooming", "Men's Key Rings & Chains", "Men's Sleepwear",
    'Deodorant', 'Cuff Links', "Men's Swimwear", 'Garden Tools',
    "Men's Gift Sets & Kits", 'Bedspreads', 'Hair Removal',
    "Men's Grooming Bags", "Men's Money Clips", 'Flatware', "Men's Hats",
    "Face Makeup", "Tech Accessories", "Lip Treatments", "Tables", "Bookcases",
    "Girls", "Hair Accessories", "Men's Watches & Jewelry",
    "Food Storage Containers", "Charms & Pendants", "Nail Treatments",
    "Activewear", "Hammocks & Swings", "Chairs", "Jewelry Storage", "Desk Lamps",
    "Face Moisturizers", "Makeup Tools", "Storage & Shelves", "Shower Curtains",
    "Necklaces", "Earrings", "Men's Necklaces", "Socks", "Scarves", 'Handbags',
    "Backpacks", "Shoulder Bags", "Men's Bags",
}

In [ ]:
# --- 2b. Consolidate fine-grained labels into broad categories ---
category_map = {
    # Jeans
    'Boyfriend Jeans': 'Jeans', 'Straight Leg Jeans': 'Jeans', 'Skinny Jeans': 'Jeans',
    'Flared Jeans': 'Jeans', 'Wide Leg Jeans': 'Jeans', 'Bootcut Jeans': 'Jeans',
    "Men's Jeans": 'Jeans',
    # Shoes
    'Sandals': 'Shoes', 'Ankle Booties': 'Shoes', 'Sneakers': 'Shoes',
    'Over-The-Knee Boots': 'Shoes', 'Knee High Boots': 'Shoes', "Men's Flip Flops": 'Shoes',
    'Slippers': 'Shoes', 'Clogs': 'Shoes', 'Mid Calf Boots': 'Shoes', "Men's Slippers": 'Shoes',
    'Loafers & Moccasins': 'Shoes', "Men's Oxfords": 'Shoes', "Men's Dress Shoes": 'Shoes',
    'Flip Flop': 'Shoes', "Men's Boots": 'Shoes', 'Athletic Shoes': 'Shoes',
    "Men's Work Boots": 'Shoes', "Men's Athletic Shoes": 'Shoes', "Men's Sandals": 'Shoes',
    "Men's Loafers & Moccasins": 'Shoes', 'Boots': 'Shoes', "Men's Sneakers": 'Shoes',
    'Pumps': 'Shoes', 'Flats': 'Shoes', 'Oxfords': 'Shoes', 'Flip Flops': 'Shoes',
    "Men's Shoes": 'Shoes',
    # Skirt
    'Knee Length Skirts': 'Skirt', 'Activewear Skirts': 'Skirt', 'Mini Skirts': 'Skirt',
    'Long Skirts': 'Skirt', 'Skirts': 'Skirt',
    # Dress
    'Rompers': 'Dress', 'Cocktail Dress': 'Dress', 'Gowns': 'Dress', 'Jumpsuit': 'Dress',
    'Wedding Dresses': 'Dress', 'Day Dresses': 'Dress', 'Cocktail Dresses': 'Dress',
    'Jumpsuits': 'Dress', 'Dresses': 'Dress',
    # Jacket
    'Cardigan': 'Jacket', 'Vests': 'Jacket', "Men's Coats": 'Jacket',
    'Activewear Jackets': 'Jacket', 'Coats': 'Jacket', 'Blazers': 'Jacket',
    'Cardigans': 'Jacket', "Men's Jackets": 'Jacket', "Men's Vests": 'Jacket',
    "Men's Sportcoats & Blazers": 'Jacket', "Men's Outerwear": 'Jacket',
    "Men's Activewear Jackets": 'Jacket', 'Jackets': 'Jacket',
    # Pants
    'Capri & Cropped Pants': 'Pants', 'Activewear Pants': 'Pants', "Men's Dress Pants": 'Pants',
    "Men's Casual Pants": 'Pants', "Men's Activewear Pants": 'Pants', "Men's Pants": 'Pants',
    # Shirt
    'Tunics': 'Shirt', "Men's Shirts": 'Shirt', 'Sweatshirts': 'Shirt', 'Blouses': 'Shirt',
    'Tops': 'Shirt', 'Sweaters': 'Shirt', 'T-Shirts': 'Shirt', "Men's Sweatshirts": 'Shirt',
    "Men's T-Shirts": 'Shirt', "Men's Casual Shirts": 'Shirt', "Men's Dress Shirts": 'Shirt',
    "Men's Polos": 'Shirt', "Men's Sweaters": 'Shirt',
    # Tank Top
    'Camisoles': 'Tank Top', 'Activewear Tops': 'Tank Top', "Men's Activewear": 'Tank Top',
    'Tank Tops': 'Tank Top', 'Activewear Tank Tops': 'Tank Top', "Men's Tank Tops": 'Tank Top',
    "Men's Activewear Tops": 'Tank Top',
    # Shorts
    'Activewear Shorts': 'Shorts', "Men's Shorts": 'Shorts', "Men's Activewear Shorts": 'Shorts',
    # Hoodie
    "Men's Hoodies": 'Hoodie', 'Hoodies': 'Hoodie',
    # Suits
    "Men's Suits": 'Suits',
}

In [ ]:
# --- 2c. Apply cleaning ---
df_clean = df[~df["category"].isin(remove_categories)].copy()
df_clean["category"] = df_clean["category"].replace(category_map)

# Derive outfit_id from item_ID (everything before the underscore)
df_clean["outfit_id"] = df_clean["item_ID"].astype(str).str.split("_").str[0]

# Drop outfits with fewer than 2 items — a single item has no pairing to learn
counts = df_clean["outfit_id"].value_counts()
valid = counts[counts >= 2].index
df_clean = df_clean[df_clean["outfit_id"].isin(valid)].copy()

# Standardize the column name used downstream
df_clean = df_clean.rename(columns={"item_ID": "item_id"})

print(f"Rows: {len(df):,} -> {len(df_clean):,}  (removed {len(df) - len(df_clean):,})")
print(f"Categories: {df['category'].nunique()} -> {df_clean['category'].nunique()}")
print(f"Outfits: {df_clean['outfit_id'].nunique():,}")
print("\nFinal category counts:")
print(df_clean["category"].value_counts())

## 3. Derive sub-genres (for Model 2)

Model 1 works on the broad categories above. Model 2 needs finer labels. We derive a **sub-genre** for each item by scanning its text description for known style keywords (`Skinny`, `Floral`, `Platform`, ...) and prepending the matched style to the category. If no keyword matches, we fall back to the plain category.

> **Limitation (worth knowing):** this is keyword matching, not learned NLP. An item described "destroyed boyfriend" without a listed keyword falls back to its category. A future version could use embeddings of the text instead. We note this in the README's limitations.

In [ ]:
STYLE_KEYWORDS = [
    'Skinny', 'Slim', 'Straight', 'Wide Leg', 'Boyfriend', 'Bootcut', 'Flared', 'Cropped',
    'Floral', 'Maxi', 'Mini', 'Midi', 'Wrap', 'Bodycon', 'Cocktail', 'Slip', 'Sundress',
    'Crop', 'Oversized', 'Fitted', 'Striped', 'Graphic', 'Lace', 'Off Shoulder',
    'Heel', 'Flat', 'Platform', 'Ankle', 'Knee High', 'Block Heel', 'Stiletto', 'Wedge',
    'Denim', 'Leather', 'Trench', 'Puffer', 'Blazer', 'Bomber',
    'Knit', 'Woven', 'Printed', 'Solid', 'Plaid', 'Checkered', 'Ruffle',
]
# Longest first so "Wide Leg" matches before "Wide"
STYLE_KEYWORDS = sorted(set(STYLE_KEYWORDS), key=lambda x: -len(x))

def extract_subgenre(text, category):
    if not isinstance(text, str) or not text.strip():
        return category
    text_title = text.title()
    for style in STYLE_KEYWORDS:
        if style in text_title:
            return f"{style} {category}"
    return category  # fall back to category if no style keyword found

df_clean["sub_genre"] = df_clean.apply(
    lambda r: extract_subgenre(r["text"], r["category"]), axis=1
)
print("Unique sub-genres:", df_clean["sub_genre"].nunique())
df_clean[["category", "sub_genre", "text"]].head(10)

## 4. Train / test split

Critically, **we split by `outfit_id`, not by row.** If two items from the same outfit landed in different splits, the model would "see the answer" during training — data leakage that inflates accuracy. Splitting whole outfits keeps the evaluation honest.

In [ ]:
outfit_ids = df_clean["outfit_id"].unique()
train_ids, test_ids = train_test_split(outfit_ids, test_size=0.2, random_state=RANDOM_STATE)

train_df = df_clean[df_clean["outfit_id"].isin(train_ids)]
test_df  = df_clean[df_clean["outfit_id"].isin(test_ids)]

print(f"Train outfits: {len(train_ids):,}  |  Test outfits: {len(test_ids):,}")

In [ ]:
# Group rows back into outfits: outfit_id -> list of item dicts
def build_outfits(frame):
    outfits = defaultdict(list)
    for _, row in frame.iterrows():
        outfits[row["outfit_id"]].append({
            "item_id":   row["item_id"],
            "category":  row["category"],
            "sub_genre": row["sub_genre"],
            "text":      row["text"],
        })
    return outfits

train_outfits = build_outfits(train_df)
test_outfits  = build_outfits(test_df)
print("Built", len(train_outfits), "train outfits and", len(test_outfits), "test outfits")

## 5. Build the co-occurrence maps

The core learning step. For every outfit, we count how often each pair of labels appears together. `co_occ[A][B]` = number of training outfits where A and B both appear. We build one map keyed on **category** (Model 1) and one keyed on **sub-genre** (Model 2).

In [ ]:
def build_cooccurrence(outfits, key):
    co = defaultdict(Counter)
    for items in outfits.values():
        labels = [it[key] for it in items]
        for a in labels:
            for b in labels:
                if a != b:
                    co[a][b] += 1
    return co

co_cat = build_cooccurrence(train_outfits, "category")    # Model 1
co_sub = build_cooccurrence(train_outfits, "sub_genre")   # Model 2

print("Model 1 — categories paired with 'Jeans':")
print(co_cat["Jeans"].most_common(5))
print("\nModel 2 — sub-genres paired with 'Skinny Jeans':")
print(co_sub.get("Skinny Jeans", Counter()).most_common(5))

## 6. Recommendation functions

**Model 1** ranks categories by how often they co-occur with the input, excluding categories the user already has.

**Model 2** does the same on sub-genres, plus one key fix: it **blocks any sub-genre from a category the user already has.** If you input `Skinny Jeans` (category `Jeans`), it won't recommend `Wide Leg Jeans` — it'll move on to a shirt, shoes, etc. This is what prevents Model 1's "dress → recommends pants" problem.

In [ ]:
# Fast lookup: sub_genre -> category (built once, no dataframe scans in the loop)
subgenre_to_category = (
    df_clean.drop_duplicates("sub_genre").set_index("sub_genre")["category"].to_dict()
)

def recommend_category(input_categories, top_k=5):
    scores = Counter()
    for c in input_categories:
        scores.update(co_cat.get(c, Counter()))
    for c in input_categories:
        scores.pop(c, None)
    return scores.most_common(top_k)

def recommend_subgenre(input_subgenres, top_k=5):
    scores = Counter()
    for sg in input_subgenres:
        scores.update(co_sub.get(sg, Counter()))
    for sg in input_subgenres:
        scores.pop(sg, None)
    # Block sub-genres from a category the input already covers
    have = {subgenre_to_category.get(sg) for sg in input_subgenres}
    for sg in [s for s in scores if subgenre_to_category.get(s) in have]:
        scores.pop(sg, None)
    return scores.most_common(top_k)

## 7. Evaluation — with baseline and top-1/3/5

This is the heart of an honest write-up. We use a **leave-one-out** test: for each test outfit, hide one item and ask the model to predict its category from the rest. We measure whether the true category appears in the model's top-K.

Three things make this a real benchmark rather than a single soft number:

1. **Popularity baseline** — always guess the most common categories, ignoring the input. If a model can't beat this, it hasn't learned anything. The *gap* between model and baseline is the real signal.
2. **Top-1, Top-3, Top-5 together** — top-5 over ~8 categories is easy; top-1 is honest. Reporting all three shows the difficulty curve.
3. **Same protocol for both models** — so we can finally say whether Model 2's sub-genres beat Model 1's categories.

We seed the hidden-item choice so the numbers are reproducible.

In [ ]:
# Most common categories overall (the popularity baseline's fixed guess)
popular_categories = [c for c, _ in train_df["category"].value_counts().items()]

def evaluate(predict_fn, ks=(1, 3, 5)):
    """predict_fn(held_out_items, k) -> list of predicted CATEGORY names (length k)."""
    rng = random.Random(RANDOM_STATE)  # reproducible hidden-item choice
    hits = {k: 0 for k in ks}
    total = 0
    for items in test_outfits.values():
        if len(items) < 2:
            continue
        hidden = rng.choice(items)
        held = [it for it in items if it["item_id"] != hidden["item_id"]]
        for k in ks:
            preds = predict_fn(held, k)
            if hidden["category"] in preds[:k]:
                hits[k] += 1
        total += 1
    return {k: hits[k] / total for k in ks}, total

# --- prediction adapters: each returns a list of predicted CATEGORIES ---
def predict_popularity(held, k):
    return popular_categories[:k]

def predict_model1(held, k):
    cats = [it["category"] for it in held]
    return [c for c, _ in recommend_category(cats, top_k=k)]

def predict_model2(held, k):
    subs = [it["sub_genre"] for it in held]
    recs = recommend_subgenre(subs, top_k=k)
    # map predicted sub-genres back to categories (dedupe, keep order)
    seen, out = set(), []
    for sg, _ in recs:
        c = subgenre_to_category.get(sg)
        if c and c not in seen:
            seen.add(c); out.append(c)
    return out

base, n   = evaluate(predict_popularity)
m1,   _   = evaluate(predict_model1)
m2,   _   = evaluate(predict_model2)

print(f"Evaluated on {n:,} test outfits\n")
header = f"{'Model':<22}{'Top-1':>8}{'Top-3':>8}{'Top-5':>8}"
print(header); print("-" * len(header))
for name, res in [("Popularity baseline", base), ("Model 1 (category)", m1), ("Model 2 (sub-genre)", m2)]:
    print(f"{name:<22}{res[1]:>7.1%}{res[3]:>8.1%}{res[5]:>8.1%}")

In [ ]:
# Visualize the comparison
import numpy as np
labels = ["Top-1", "Top-3", "Top-5"]
x = np.arange(len(labels)); w = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w, [base[1], base[3], base[5]], w, label="Popularity baseline")
ax.bar(x,     [m1[1],   m1[3],   m1[5]],   w, label="Model 1 (category)")
ax.bar(x + w, [m2[1],   m2[3],   m2[5]],   w, label="Model 2 (sub-genre)")
ax.set_ylabel("Accuracy"); ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Missing-category prediction accuracy"); ax.legend()
ax.set_ylim(0, 1)
for i, v in enumerate([base[1], base[3], base[5]]): ax.text(i - w, v + .01, f"{v:.0%}", ha="center", fontsize=8)
for i, v in enumerate([m1[1], m1[3], m1[5]]):       ax.text(i,     v + .01, f"{v:.0%}", ha="center", fontsize=8)
for i, v in enumerate([m2[1], m2[3], m2[5]]):       ax.text(i + w, v + .01, f"{v:.0%}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

## 8. Build & display a complete outfit

Given one item, fill the remaining slots with real items pulled from the recommended sub-genres (one item per category, so the outfit has variety).

In [ ]:
def build_complete_outfit(input_subgenres, top_k=10, n_items=3):
    recs = recommend_subgenre(input_subgenres, top_k=top_k)
    used = {subgenre_to_category.get(sg) for sg in input_subgenres}
    chosen = []
    for sg, _ in recs:
        cat = subgenre_to_category.get(sg)
        if cat in used:
            continue
        pool = df_clean[df_clean["sub_genre"] == sg]
        if len(pool):
            chosen.append(pool.sample(1, random_state=RANDOM_STATE).iloc[0])
            used.add(cat)
        if len(chosen) == n_items:
            break
    return chosen

def show_recommendations(items):
    if not items:
        print("No recommendations found — try another sub-genre."); return
    fig, axes = plt.subplots(1, len(items), figsize=(4 * len(items), 4))
    if len(items) == 1: axes = [axes]
    for ax, item in zip(axes, items):
        img = Image.open(io.BytesIO(ast.literal_eval(item["image"])["bytes"]))
        ax.imshow(img); ax.axis("off")
        ax.set_title(f'{item["sub_genre"]}\n{item["text"][:40]}', fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# Example: start from "Skinny Jeans" and complete the outfit
example = build_complete_outfit(["Skinny Jeans"])
print("Recommended outfit for input 'Skinny Jeans':")
show_recommendations(example)

## 9. Interactive demo

Run this cell, type a sub-genre (e.g. `Floral Dress`, `Wide Leg Jeans`, `Leather Jacket`), and see a generated outfit.

In [ ]:
available = sorted(df_clean["sub_genre"].unique())
print(f"{len(available)} sub-genres available. Examples:", available[:15])

user_input = input("\nEnter a sub-genre: ").strip().title()
if user_input not in df_clean["sub_genre"].values:
    matches = [sg for sg in available if user_input in sg]
    if matches:
        user_input = matches[0]
        print("Closest match:", user_input)
    else:
        print("No match. Pick one from the list above."); user_input = None

if user_input:
    show_recommendations(build_complete_outfit([user_input]))

## 10. Limitations & next steps

**What this is.** A count-based co-occurrence recommender — a strong, interpretable baseline that learns real pairing patterns directly from data.

**What it isn't / where it's weak.**
- Co-occurrence favors popular items: common categories (Shirt, Shoes) get recommended a lot regardless of true compatibility. *Weighting by PMI / lift would surface distinctive pairings.*
- Sub-genres come from keyword matching, not learned text understanding, so coverage is uneven.
- Category-level evaluation doesn't fully reward Model 2's finer detail — it measures "what type of item fits," not "which specific style."
- No notion of visual style/color compatibility. *CLIP image embeddings are the natural next step.*

**Next steps:** PMI-weighted scoring, a proper recall@k for sub-genres, CLIP-based visual compatibility, and a hosted Streamlit/Gradio demo.
